In [17]:
!pip install -q datasets tensorflow

In [18]:
import tensorflow as tf
import numpy as np
import pickle
from datasets import load_dataset

In [19]:
dataset = load_dataset("roneneldan/TinyStories")

texts = dataset["train"]["text"]

target_length = 2_000_000  # 10 lakh characters

combined_text = ""
for t in texts:
    combined_text += t + "\n"
    if len(combined_text) >= target_length:
        break

combined_text = combined_text[:target_length].lower()

print("Final text length:", len(combined_text))

Final text length: 2000000


In [20]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(
    num_words=20000,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([combined_text])

vocab_size = 20000
print("Vocab size:", vocab_size)

Vocab size: 20000


In [21]:
max_len = 64

tokens = tokenizer.texts_to_sequences([combined_text])[0]

sequences = []
for i in range(0, len(tokens) - max_len):
    sequences.append(tokens[i:i+max_len+1])

sequences = np.array(sequences)

X = sequences[:, :-1]
y = sequences[:, 1:]

print("Total sequences:", len(X))

Total sequences: 389957


In [22]:
X = X[:120000]
y = y[:120000]

In [23]:
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()

        self.att = tf.keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim,
            dropout=dropout
        )

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="gelu"),
            tf.keras.layers.Dense(embed_dim),
        ])

        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)

        self.dropout1 = tf.keras.layers.Dropout(dropout)
        self.dropout2 = tf.keras.layers.Dropout(dropout)

    def call(self, x, training=False):
        attn_output = self.att(
            x, x,
            use_causal_mask=True
        )

        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)

        return self.layernorm2(out1 + ffn_output)

In [24]:
class TokenAndPositionEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = tf.keras.layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = tf.keras.layers.Embedding(max_len, embed_dim)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        positions = tf.range(0, seq_len, 1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [25]:
def build_mini_gpt(
    vocab_size,
    max_len,
    embed_dim=128,
    num_heads=4,
    ff_dim=256,
    num_layers=2,
    dropout=0.1
):

    inputs = tf.keras.layers.Input(shape=(max_len,))

    x = TokenAndPositionEmbedding(max_len, vocab_size, embed_dim)(inputs)

    for _ in range(num_layers):
        x = TransformerBlock(embed_dim, num_heads, ff_dim, dropout)(x)

    x = tf.keras.layers.LayerNormalization(epsilon=1e-6)(x)

    outputs = tf.keras.layers.Dense(vocab_size)(x)  # no softmax

    return tf.keras.Model(inputs=inputs, outputs=outputs)

In [26]:
model = build_mini_gpt(
    vocab_size=vocab_size,
    max_len=max_len
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=5e-5,
        weight_decay=1e-4,
        clipnorm=1.0
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_1  │ (None, 64, 128)        │     2,568,192 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ (None, 64, 128)        │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_3             │ (None, 64, 128)        │       330,240 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_9           │ (None, 64, 128)        │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 64, 20000)      │     2,580,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,808,928 (22.16 MB)

 Trainable params: 5,808,928 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    X,
    y,
    batch_size=64,
    epochs=15
)

Epoch 1/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 99s 43ms/step - accuracy: 0.0712 - loss: 7.2738
Epoch 2/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.2176 - loss: 4.5393
Epoch 3/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.2737 - loss: 3.8876
Epoch 4/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.3199 - loss: 3.4433
Epoch 5/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.3647 - loss: 3.0723
Epoch 6/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.4076 - loss: 2.7538
Epoch 7/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.4509 - loss: 2.4743
Epoch 8/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.4934 - loss: 2.2272
Epoch 9/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.5344 - loss: 2.0102
Epoch 10/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.5731 - loss: 1.8158
Epoch 11/15
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 45ms/step - accuracy: 0.6078 - loss: 1.6495
Epoch 12

In [12]:
def generate_text(seed_text, next_words=50, temperature=0.8):

    seed_text = seed_text.lower()

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = token_list[-max_len:]
        token_list = tf.keras.preprocessing.sequence.pad_sequences(
            [token_list],
            maxlen=max_len,
            padding="pre"
        )

        predictions = model.predict(token_list, verbose=0)
        logits = predictions[0, -1] / temperature

        probs = tf.nn.softmax(logits).numpy()
        next_index = np.random.choice(len(probs), p=probs)

        word = tokenizer.index_word.get(next_index, "")
        seed_text += " " + word

    return seed_text

In [27]:
print(generate_text("i go to", 50, 0.7))

i go to   batter   etc    anchor   attention  shyly   lion signed watered remains   crumbling kitty bitten    instead     healthier  regreted    pitch   haha      luckily


In [14]:
model.save("mini_gpt_tinystories_1M.keras")

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [15]:
from google.colab import files

files.download("mini_gpt_tinystories_1M.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
files.download("tokenizer.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>